[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/096.%20The%20Resilient%20Network%20Design%20%28Robust%20Optimization%29/P96-Tier-2_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 96. The Resilient Network Design (Robust Optimization)
## Tier 2: The Classic Heuristic (Sweep Algorithm)

### Key assumptions
- Geographic diversity improves network resilience against localized disruptions
- Facilities can be evaluated based on their angular positions from the geographic center
- A systematic sweep through angular positions ensures comprehensive coverage
- Resilience can be evaluated through scenario-based analysis
- Cost constraints limit the number of facilities that can be selected

### Approach (step-by-step)
1. **Compute geographic center** of all potential facility locations
2. **Sort facilities by polar angle** from the geographic center
3. **Sweep through starting positions** and select facilities in angular order
4. **Evaluate each selection** against all disruption scenarios
5. **Select best configuration** that meets resilience requirements at minimum cost

### What to look for in the results
- Geographically diverse facility selection
- Service coverage levels across different scenarios
- Trade-offs between geographical diversity and cost efficiency
- Comparison with exact mathematical optimization results

### Concrete example (from the source)
The sweep algorithm selects facilities from 8 candidates at coordinates:
- Facilities: (10,20), (30,15), (25,35), (45,30), (15,50), (40,10), (35,45), (20,40)
- Demand zones: (15,25), (35,20), (25,40), (40,35)
- Objective: Maximize geographical diversity while keeping transport costs below $50,000

In [1]:
from typing import Tuple, List, Dict, Set

# Import required packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Set
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Define data structures for the sweep algorithm
@dataclass
class Facility:
    """Represents a potential facility location"""
    id: int
    x: float
    y: float
    capacity: int
    fixed_cost: float
    
    def distance_to(self, other) -> float:
        """Calculate Euclidean distance to another point"""
        return np.sqrt((self.x - other.x)**2 + (self.y - other.y)**2)

@dataclass
class DemandZone:
    """Represents a demand zone"""
    id: int
    x: float
    y: float
    demand: float
    
    def distance_to(self, facility: Facility) -> float:
        """Calculate distance to a facility"""
        return np.sqrt((self.x - facility.x)**2 + (self.y - facility.y)**2)

@dataclass
class Scenario:
    """Represents a disruption scenario"""
    id: int
    probability: float
    facility_failures: List[int]  # Facility IDs that fail
    route_blocks: Set[Tuple[int, int]]  # (facility_id, demand_id) blocked routes
    demand_multipliers: dict  # demand_id -> multiplier

@dataclass
class SweepResult:
    """Results from sweep algorithm evaluation"""
    selected_facilities: List[Facility]
    total_cost: float
    worst_service_coverage: float
    expected_service_coverage: float
    geographical_diversity: float

In [ ]:
# Create the concrete example from the source
def create_example_problem():
    """Create the example problem from the source material"""
    
    # Define 8 potential facilities
    facilities = [
        Facility(1, 10, 20, 1000, 200000),   # Location 1
        Facility(2, 30, 15, 1200, 250000),   # Location 2
        Facility(3, 25, 35, 1100, 230000),   # Location 3
        Facility(4, 45, 30, 1300, 280000),   # Location 4
        Facility(5, 15, 50, 900, 180000),    # Location 5
        Facility(6, 40, 10, 1400, 300000),   # Location 6
        Facility(7, 35, 45, 1150, 240000),   # Location 7
        Facility(8, 20, 40, 1050, 210000),    # Location 8
    ]
    
    # Define 4 demand zones
    demand_zones = [
        DemandZone(1, 15, 25, 200),  # Zone 1
        DemandZone(2, 35, 20, 300),  # Zone 2
        DemandZone(3, 25, 40, 250),  # Zone 3
        DemandZone(4, 40, 35, 350),  # Zone 4
    ]
    
    # Define disruption scenarios
    scenarios = [
        # Normal scenario (60% probability)
        Scenario(
            id=1,
            probability=0.6,
            facility_failures=[],
            route_blocks=set(),
            demand_multipliers={1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0}
        ),
        # Hurricane scenario (25% probability)
        Scenario(
            id=2,
            probability=0.25,
            facility_failures=[2, 6],  # Facilities 2 and 6 fail
            route_blocks={(1, 3), (4, 2)},  # Some routes blocked
            demand_multipliers={1: 1.5, 2: 0.7, 3: 1.3, 4: 0.8}
        ),
        # Flood scenario (15% probability)
        Scenario(
            id=3,
            probability=0.15,
            facility_failures=[5],  # Facility 5 fails
            route_blocks={(3, 1), (7, 4)},  # Some routes blocked
            demand_multipliers={1: 0.8, 2: 1.2, 3: 0.9, 4: 1.1}
        )
    ]
    
    return facilities, demand_zones, scenarios

# Create the problem instance
facilities, demand_zones, scenarios = create_example_problem()

print("Problem Setup:")
print(f"- Facilities: {len(facilities)} potential locations")
print(f"- Demand zones: {len(demand_zones)} service areas")
print(f"- Scenarios: {len(scenarios)} disruption scenarios")
print(f"- Cost constraint: $50,000 maximum transport cost")
print(f"- Service requirement: At least 80% coverage in worst case")

Problem Setup:
- Facilities: 8 potential locations
- Demand zones: 4 service areas
- Scenarios: 3 disruption scenarios
- Cost constraint: $50,000 maximum transport cost
- Service requirement: At least 80% coverage in worst case


In [ ]:
# Implement the sweep algorithm core functions
class ResilientNetworkSweep:
    """Sweep algorithm for resilient network design"""
    
    def __init__(self, facilities: List[Facility], demand_zones: List[DemandZone], 
                 scenarios: List[Scenario], max_facilities: int = 3):
        self.facilities = facilities
        self.demand_zones = demand_zones
        self.scenarios = scenarios
        self.max_facilities = max_facilities
        
        # Compute geographic center of all facilities
        self.center_x = np.mean([f.x for f in facilities])
        self.center_y = np.mean([f.y for f in facilities])
        self.center = (self.center_x, self.center_y)
        
        # Sort facilities by polar angle from center
        self.angles = []
        for i, facility in enumerate(facilities):
            angle = np.arctan2(facility.y - self.center_y, facility.x - self.center_x)
            self.angles.append((angle, i))
        self.angles.sort()  # Sort by angle
        
    def calculate_geographical_diversity(self, selected_facilities: List[Facility]) -> float:
        """Calculate geographical diversity score based on angular separation"""
        if len(selected_facilities) < 2:
            return 0.0
        
        # Get angles of selected facilities
        selected_angles = []
        for facility in selected_facilities:
            angle = np.arctan2(facility.y - self.center_y, facility.x - self.center_x)
            selected_angles.append(angle)
        
        # Sort angles
        selected_angles.sort()
        
        # Calculate angular separations
        separations = []
        n = len(selected_angles)
        for i in range(n):
            next_angle = selected_angles[(i + 1) % n]
            current_angle = selected_angles[i]
            if i == n - 1:  # Wrap around
                separation = (2 * np.pi - current_angle) + next_angle
            else:
                separation = next_angle - current_angle
            separations.append(separation)
        
        # Diversity score: higher when angles are more evenly distributed
        ideal_separation = 2 * np.pi / n
        variance = np.var([sep - ideal_separation for sep in separations])
        diversity = max(0, 1 - variance / (ideal_separation ** 2))
        
        return diversity
    
    def calculate_service_coverage(self, selected_facilities: List[Facility], 
                                  scenario: Scenario) -> float:
        """Calculate service coverage for a given scenario"""
        # Filter available facilities (not failed)
        available_facilities = [f for f in selected_facilities 
                               if f.id not in scenario.facility_failures]
        
        if not available_facilities:
            return 0.0
        
        total_demand = 0
        served_demand = 0
        
        for demand in self.demand_zones:
            actual_demand = demand.demand * scenario.demand_multipliers.get(demand.id, 1.0)
            total_demand += actual_demand
            
            # Find best service from available facilities
            best_service = 0
            for facility in available_facilities:
                # Check if route is blocked
                if (facility.id, demand.id) in scenario.route_blocks:
                    continue
                
                # Calculate service based on distance and capacity
                distance = facility.distance_to(demand)
                service_factor = 1 / (1 + distance * 0.01)  # Distance penalty
                service = min(facility.capacity, actual_demand) * service_factor
                best_service = max(best_service, service)
            
            served_demand += best_service
        
        return served_demand / total_demand if total_demand > 0 else 0
    
    def calculate_transport_cost(self, selected_facilities: List[Facility], 
                               scenario: Scenario) -> float:
        """Calculate total transport cost for a scenario"""
        available_facilities = [f for f in selected_facilities 
                               if f.id not in scenario.facility_failures]
        
        total_cost = 0
        
        for demand in self.demand_zones:
            actual_demand = demand.demand * scenario.demand_multipliers.get(demand.id, 1.0)
            min_cost = float('inf')
            
            for facility in available_facilities:
                if (facility.id, demand.id) in scenario.route_blocks:
                    continue
                
                distance = facility.distance_to(demand)
                cost = distance * 0.1 * actual_demand  # $0.1 per unit distance per unit demand
                min_cost = min(min_cost, cost)
            
            if min_cost < float('inf'):
                total_cost += min_cost
        
        return total_cost
    
    def evaluate_selection(self, selected_facilities: List[Facility]) -> SweepResult:
        """Evaluate a facility selection against all scenarios"""
        if len(selected_facilities) < 2:
            return SweepResult(selected_facilities, float('inf'), 0, 0, 0)
        
        # Calculate fixed costs
        fixed_cost = sum(f.fixed_cost for f in selected_facilities)
        
        # Evaluate across all scenarios
        total_expected_cost = 0
        worst_service = 1.0
        service_coverages = []
        
        for scenario in self.scenarios:
            service_coverage = self.calculate_service_coverage(selected_facilities, scenario)
            transport_cost = self.calculate_transport_cost(selected_facilities, scenario)
            
            total_cost = fixed_cost + transport_cost
            total_expected_cost += scenario.probability * total_cost
            
            worst_service = min(worst_service, service_coverage)
            service_coverages.append(service_coverage)
        
        expected_service = np.mean(service_coverages)
        geographical_diversity = self.calculate_geographical_diversity(selected_facilities)
        
        return SweepResult(
            selected_facilities=selected_facilities,
            total_cost=total_expected_cost,
            worst_service_coverage=worst_service,
            expected_service_coverage=expected_service,
            geographical_diversity=geographical_diversity
        )
    
    def sweep_search(self, max_transport_cost: float = 50000) -> SweepResult:
        """Perform sweep search to find best facility selection"""
        best_result = None
        best_cost = float('inf')
        
        n = len(self.facilities)
        
        # Sweep through all starting positions
        for start_idx in range(n):
            # Select facilities in sweep order
            selected = []
            for k in range(self.max_facilities):
                facility_idx = self.angles[(start_idx + k) % n][1]
                selected.append(self.facilities[facility_idx])
            
            # Evaluate this selection
            result = self.evaluate_selection(selected)
            
            # Check if it meets constraints and is better
            if (result.worst_service_coverage >= 0.8 and 
                result.total_cost <= max_transport_cost and 
                result.total_cost < best_cost):
                best_result = result
                best_cost = result.total_cost
        
        return best_result if best_result is not None else SweepResult([], float('inf'), 0, 0, 0)

In [ ]:
try:
    # Run the sweep algorithm
    def run_sweep_algorithm():
        """Execute the sweep algorithm and analyze results"""
        
        print("=" * 60)
        print("RESILIENT NETWORK SWEEP ALGORITHM")
        print("=" * 60)
        
        # Initialize sweep algorithm
        sweep = ResilientNetworkSweep(facilities, demand_zones, scenarios, max_facilities=3)
        
        print(f"\nGeographic Center: ({sweep.center_x:.1f}, {sweep.center_y:.1f})")
        print(f"\nFacilities sorted by angle from center:")
        for angle, idx in sweep.angles:
            facility = facilities[idx]
            angle_deg = np.degrees(angle)
            print(f"  Facility {facility.id}: ({facility.x}, {facility.y}) - Angle: {angle_deg:.1f}°")
        
        # Perform sweep search
        print(f"\nPerforming sweep search with max {sweep.max_facilities} facilities...")
        best_result = sweep.sweep_search(max_transport_cost=50000)
        
        # Display results
        print("\n" + "=" * 60)
        print("SWEEP ALGORITHM RESULTS")
        print("=" * 60)
        
        if best_result.total_cost == float('inf'):
            print("No feasible solution found within constraints!")
            return None
        
        print(f"\nSELECTED FACILITIES:")
        for facility in best_result.selected_facilities:
            print(f"  Facility {facility.id}: ({facility.x}, {facility.y}) - Cost: ${facility.fixed_cost:,}")
        
        print(f"\nPERFORMANCE METRICS:")
        print(f"  Expected Total Cost: ${best_result.total_cost:,.0f}")
        print(f"  Worst-Case Service Coverage: {best_result.worst_service_coverage:.1%}")
        print(f"  Expected Service Coverage: {best_result.expected_service_coverage:.1%}")
        print(f"  Geographical Diversity Score: {best_result.geographical_diversity:.3f}")
        
        # Check constraints
        print(f"\nCONSTRAINT CHECK:")
        if best_result.worst_service_coverage >= 0.8:
            print(f"  ✓ Service coverage requirement met ({best_result.worst_service_coverage:.1%} ≥ 80%)")
        else:
            print(f"  ✗ Service coverage requirement NOT met ({best_result.worst_service_coverage:.1%} < 80%)")
        
        if best_result.total_cost <= 50000:
            print(f"  ✓ Cost constraint met (${best_result.total_cost:,.0f} ≤ $50,000)")
        else:
            print(f"  ✗ Cost constraint exceeded (${best_result.total_cost:,.0f} > $50,000)")
        
        return sweep, best_result
    
    # Run the algorithm
    sweep, best_result = run_sweep_algorithm()
except Exception as e:
    print(f'Error in cell: {e}')

RESILIENT NETWORK SWEEP ALGORITHM

Geographic Center: (27.5, 30.6)

Facilities sorted by angle from center:
  Facility 1: (10, 20) - Angle: -148.7°
  Facility 2: (30, 15) - Angle: -80.9°
  Facility 6: (40, 10) - Angle: -58.8°
  Facility 4: (45, 30) - Angle: -2.0°
  Facility 7: (35, 45) - Angle: 62.4°
  Facility 3: (25, 35) - Angle: 119.7°
  Facility 5: (15, 50) - Angle: 122.8°
  Facility 8: (20, 40) - Angle: 128.7°

Performing sweep search with max 3 facilities...

SWEEP ALGORITHM RESULTS
No feasible solution found within constraints!
Error in cell: cannot unpack non-iterable NoneType object


In [ ]:
try:
    # Detailed scenario analysis
    def analyze_scenarios_detailed():
        """Analyze performance across all scenarios in detail"""
        
        if best_result is None:
            print("No solution to analyze")
            return
        
        print("\n" + "=" * 60)
        print("DETAILED SCENARIO ANALYSIS")
        print("=" * 60)
        
        for scenario in scenarios:
            print(f"\nScenario {scenario.id}:")
            print(f"  Probability: {scenario.probability}")
            print(f"  Failed Facilities: {scenario.facility_failures}")
            print(f"  Blocked Routes: {scenario.route_blocks}")
            print(f"  Demand Multipliers: {scenario.demand_multipliers}")
            
            # Calculate performance metrics
            service_coverage = sweep.calculate_service_coverage(best_result.selected_facilities, scenario)
            transport_cost = sweep.calculate_transport_cost(best_result.selected_facilities, scenario)
            fixed_cost = sum(f.fixed_cost for f in best_result.selected_facilities)
            total_cost = fixed_cost + transport_cost
            
            print(f"  Service Coverage: {service_coverage:.1%}")
            print(f"  Transport Cost: ${transport_cost:,.0f}")
            print(f"  Fixed Cost: ${fixed_cost:,.0f}")
            print(f"  Total Cost: ${total_cost:,.0f}")
            
            # Show demand fulfillment details
            print(f"  Demand Fulfillment:")
            for demand in demand_zones:
                actual_demand = demand.demand * scenario.demand_multipliers.get(demand.id, 1.0)
                print(f"    Zone {demand.id}: {actual_demand:.0f} units demand")
    
    # Analyze scenarios
    analyze_scenarios_detailed()
except Exception as e:
    print(f'Error in cell: {e}')

Error in cell: name 'best_result' is not defined


In [ ]:
try:
    # Create comprehensive visualization
    def create_sweep_visualization():
        """Create comprehensive visualization of sweep algorithm results"""
        
        if best_result is None:
            print("No solution to visualize")
            return
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Sweep Algorithm for Resilient Network Design', fontsize=16, fontweight='bold')
        
        # Panel 1: Geographic Layout with Sweep Pattern
        ax1 = axes[0, 0]
        ax1.set_title('Geographic Layout and Sweep Pattern', fontweight='bold')
        
        # Plot geographic center
        ax1.scatter(sweep.center_x, sweep.center_y, c='red', marker='*', s=300, 
                   alpha=0.8, edgecolors='black', linewidth=2, label='Geographic Center')
        ax1.annotate('Center', (sweep.center_x, sweep.center_y), 
                    xytext=(5, 5), textcoords='offset points', fontweight='bold')
        
        # Plot all facilities with angular positions
        for angle, idx in sweep.angles:
            facility = facilities[idx]
            color = 'green' if facility in best_result.selected_facilities else 'lightgray'
            marker = 'o' if facility in best_result.selected_facilities else 'x'
            size = 300 if facility in best_result.selected_facilities else 200
            ax1.scatter(facility.x, facility.y, c=color, marker=marker, s=size, 
                       alpha=0.7, edgecolors='black', linewidth=2)
            
            # Draw angle line from center
            line_length = 15
            end_x = sweep.center_x + line_length * np.cos(angle)
            end_y = sweep.center_y + line_length * np.sin(angle)
            ax1.plot([sweep.center_x, end_x], [sweep.center_y, end_y], 
                    'k--', alpha=0.3, linewidth=1)
            
            ax1.annotate(f'F{facility.id}', (facility.x, facility.y), 
                        xytext=(5, 5), textcoords='offset points', fontweight='bold')
        
        # Plot demand zones
        for demand in demand_zones:
            ax1.scatter(demand.x, demand.y, c='blue', marker='s', s=200, 
                       alpha=0.7, edgecolors='black', linewidth=2)
            ax1.annotate(f'D{demand.id}', (demand.x, demand.y), 
                        xytext=(5, 5), textcoords='offset points', fontweight='bold')
        
        ax1.set_xlabel('X Coordinate')
        ax1.set_ylabel('Y Coordinate')
        ax1.grid(True, alpha=0.3)
        ax1.legend(loc='upper right')
        
        # Panel 2: Angular Distribution
        ax2 = axes[0, 1]
        ax2.set_title('Angular Distribution of Selected Facilities', fontweight='bold')
        
        # Get angles of selected facilities
        selected_angles = []
        for facility in best_result.selected_facilities:
            angle = np.arctan2(facility.y - sweep.center_y, facility.x - sweep.center_x)
            selected_angles.append(np.degrees(angle))
        
        # Create circular plot
        angles_rad = np.radians(selected_angles + [selected_angles[0]])  # Close the circle
        radii = [1] * len(angles_rad)
        
        ax2.plot(angles_rad, radii, 'bo-', linewidth=2, markersize=10)
        ax2.fill(angles_rad, radii, alpha=0.3)
        
        # Add facility labels
        for i, (facility, angle_deg) in enumerate(zip(best_result.selected_facilities, selected_angles)):
            angle_rad = np.radians(angle_deg)
            ax2.text(angle_rad, 1.15, f'F{facility.id}', ha='center', va='center', fontweight='bold')
        
        ax2.set_ylim(0, 1.3)
        ax2.set_theta_zero_location('E')
        ax2.set_theta_direction(1)
        ax2.set_title('Angular Distribution of Selected Facilities', fontweight='bold')
        
        # Panel 3: Scenario Performance Comparison
        ax3 = axes[1, 0]
        ax3.set_title('Performance Across Scenarios', fontweight='bold')
        
        scenario_names = ['Normal', 'Hurricane', 'Flood']
        service_coverages = []
        transport_costs = []
        
        for scenario in scenarios:
            service_coverage = sweep.calculate_service_coverage(best_result.selected_facilities, scenario)
            transport_cost = sweep.calculate_transport_cost(best_result.selected_facilities, scenario)
            service_coverages.append(service_coverage * 100)
            transport_costs.append(transport_cost)
        
        x_pos = np.arange(len(scenario_names))
        width = 0.35
        
        bars1 = ax3.bar(x_pos - width/2, service_coverages, width, label='Service Coverage (%)', alpha=0.8)
        ax3_twin = ax3.twinx()
        bars2 = ax3_twin.bar(x_pos + width/2, transport_costs, width, label='Transport Cost ($)', alpha=0.8, color='orange')
        
        ax3.set_xlabel('Scenario')
        ax3.set_ylabel('Service Coverage (%)', color='blue')
        ax3_twin.set_ylabel('Transport Cost ($)', color='orange')
        ax3.set_xticks(x_pos)
        ax3.set_xticklabels(scenario_names)
        ax3.grid(True, alpha=0.3)
        
        # Add 80% threshold line
        ax3.axhline(y=80, color='red', linestyle='--', alpha=0.7, label='80% Threshold')
        
        # Panel 4: Algorithm Convergence
        ax4 = axes[1, 1]
        ax4.set_title('Sweep Algorithm Search Process', fontweight='bold')
        
        # Show sweep process results
        sweep_results = []
        n = len(facilities)
        
        for start_idx in range(min(n, 12)):  # Show first 12 for clarity
            selected = []
            for k in range(sweep.max_facilities):
                facility_idx = sweep.angles[(start_idx + k) % n][1]
                selected.append(facilities[facility_idx])
            
            result = sweep.evaluate_selection(selected)
            if result.total_cost < float('inf'):
                sweep_results.append((start_idx, result.worst_service_coverage * 100, result.total_cost))
        
        if sweep_results:
            start_positions, coverages, costs = zip(*sweep_results)
            
            ax4.scatter(coverages, costs, alpha=0.7, s=100)
            
            # Highlight best solution
            best_idx = np.argmin(costs)
            ax4.scatter(coverages[best_idx], costs[best_idx], color='red', s=200, 
                       marker='*', edgecolors='black', linewidth=2, label='Best Solution')
            
            # Add feasibility region
            ax4.axhline(y=50000, color='orange', linestyle='--', alpha=0.7, label='Cost Limit')
            ax4.axvline(x=80, color='red', linestyle='--', alpha=0.7, label='Coverage Requirement')
            
            # Shade feasible region
            ax4.fill_between([80, 100], 50000, max(costs) * 1.1, alpha=0.2, color='green', label='Feasible Region')
        
        ax4.set_xlabel('Service Coverage (%)')
        ax4.set_ylabel('Total Cost ($)')
        ax4.grid(True, alpha=0.3)
        ax4.legend(loc='upper right')
        
        plt.tight_layout()
        plt.show()
        
        return fig
    
    # Create visualization
    fig = create_sweep_visualization()
except Exception as e:
    print(f'Error in cell: {e}')

Error in cell: name 'best_result' is not defined


In [ ]:
try:
    # Performance comparison with mathematical optimization
    def compare_with_mathematical_optimization():
        """Compare sweep algorithm performance with theoretical optimum"""
        
        print("\n" + "=" * 60)
        print("PERFORMANCE COMPARISON ANALYSIS")
        print("=" * 60)
        
        if best_result is None:
            print("No solution to compare")
            return
        
        # Theoretical analysis: what would be the optimal solution?
        print("\nALGORITHM COMPARISON:")
        print(f"\nSweep Algorithm Solution:")
        print(f"  Selected Facilities: {[f.id for f in best_result.selected_facilities]}")
        print(f"  Total Cost: ${best_result.total_cost:,.0f}")
        print(f"  Worst-Case Coverage: {best_result.worst_service_coverage:.1%}")
        print(f"  Geographical Diversity: {best_result.geographical_diversity:.3f}")
        
        # Estimate theoretical bounds
        print(f"\nTheoretical Analysis:")
        
        # Lower bound on cost (minimum fixed costs)
        min_fixed_costs = sorted([f.fixed_cost for f in facilities])[:sweep.max_facilities]
        lower_bound_cost = sum(min_fixed_costs)
        print(f"  Lower Bound Cost (cheapest {sweep.max_facilities} facilities): ${lower_bound_cost:,.0f}")
        
        # Upper bound on cost (most expensive facilities)
        max_fixed_costs = sorted([f.fixed_cost for f in facilities], reverse=True)[:sweep.max_facilities]
        upper_bound_cost = sum(max_fixed_costs)
        print(f"  Upper Bound Cost (most expensive {sweep.max_facilities} facilities): ${upper_bound_cost:,.0f}")
        
        # Quality assessment
        cost_efficiency = (best_result.total_cost - lower_bound_cost) / (upper_bound_cost - lower_bound_cost)
        print(f"\nSolution Quality Assessment:")
        print(f"  Cost Efficiency: {(1 - cost_efficiency) * 100:.1f}% (0% = worst, 100% = best)")
        print(f"  Service Coverage: {'✓' if best_result.worst_service_coverage >= 0.8 else '✗'}")
        print(f"  Cost Constraint: {'✓' if best_result.total_cost <= 50000 else '✗'}")
        
        # Algorithm advantages
        print(f"\nSweep Algorithm Advantages:")
        print(f"  ✓ Computational efficiency: O(n²) complexity vs exponential for exact methods")
        print(f"  ✓ Geographical diversity inherently optimized")
        print(f"  ✓ Transparent selection process based on angular positioning")
        print(f"  ✓ Scalable to larger problem instances")
        
        print(f"\nPotential Limitations:")
        print(f"  ⚠ May miss optimal solutions that don't follow angular patterns")
        print(f"  ⚠ Limited to selecting facilities in angular order")
        print(f"  ⚠ May not optimize cost as effectively as mathematical programming")
        
        return best_result
    
    # Compare performance
    comparison_result = compare_with_mathematical_optimization()
except Exception as e:
    print(f'Error in cell: {e}')


PERFORMANCE COMPARISON ANALYSIS
Error in cell: name 'best_result' is not defined


### Why this Tier exists vs Tier 1
This Tier 2 provides a practical heuristic approach when exact optimization becomes computationally challenging:

**Key Advantages over Tier 1:**
- **Computational efficiency**: O(n²) complexity vs exponential growth for mixed-integer programming
- **Scalability**: Can handle much larger problem instances with hundreds of facilities
- **Geographical intuition**: Leverages spatial diversity principles for resilience
- **Real-time applicability**: Suitable for dynamic decision-making environments
- **Implementation simplicity**: Easy to understand and modify for specific requirements

**When to prefer this Tier:**
- Large-scale networks where exact optimization is intractable
- Time-sensitive decision making requiring quick solutions
- Situations where geographical diversity is a primary concern
- Resource-constrained environments with limited computational power
- Initial screening before applying more sophisticated methods

### Pros / Cons vs Tier 1
**Pros:**
- Fast computation for large problem instances
- Inherent geographical diversity optimization
- Transparent and explainable selection process
- Scalable to real-world problem sizes
- Robust to data uncertainties and approximations

**Cons:**
- No guarantee of optimality
- Limited to angular-based selection patterns
- May miss cost-effective non-diverse solutions
- Less comprehensive constraint handling
- Performance depends on geographic distribution of facilities

### When to use this Tier
- **Large distribution networks** with 50+ potential facilities
- **Emergency response planning** requiring quick decisions
- **Geographically dispersed operations** where spatial diversity matters
- **Preliminary analysis** before detailed optimization
- **Real-time reconfiguration** during ongoing disruptions
- **Resource-limited environments** with computational constraints